# 01 — Variables & Data Types (Names → Objects)
### Python Basics · Part 1 of 5

Picture the front desk of a small gym. A member walks in, the clerk types her age into a form — `25` — and two lines later the program tries to work out next year's age and **crashes**. Nothing is wrong with the clerk or the maths. The program just didn't know *what kind of thing* `25` was.

That is this whole notebook, boiled down to one question you'll ask a thousand times:

> **What is this thing, and what am I allowed to do to it?**

The running example for all of Python Basics is **Iron Gym** — a small-town gym with a front desk, a fee list, a gate and an attendance sheet. Every notebook builds one more piece of it and ends with a small real-world capstone.

## The core idea: a variable is a name tag, not a box

Most people picture a variable as a box with a value inside. That picture works for about a week, then breaks. The accurate picture:

| The "box" picture | What Python really does |
|---|---|
| `age = 25` puts 25 *in a box called age* | Creates an object `25` in memory and sticks a **name tag** `age` on it |
| `b = a` copies the value | Sticks a **second tag** on the *same* object |
| `age = 26` changes what's in the box | Moves the tag `age` to a *different* object |

Most "weird" Python bugs in the first year — lists changing behind your back, `is` vs `==`, surprising default arguments — are this one misunderstanding.

> 📌 **Python is dynamically typed but strongly typed.** *Dynamically:* you never declare `int age`; a name can point at an `int` now and a `str` later. *Strongly:* Python will **not** quietly turn `"25"` into `25` for you — `"25" + 1` is an error, not `26` (JavaScript would happily give you `"251"`). Annoying on day one, a lifesaver on day 100.

## Setup

No installs. No API keys. Just Python 3.10+. The helper below runs one line of code and **shows the error instead of crashing the notebook** — we'll use it to look at mistakes safely. (Real error handling with `try`/`except` is a later topic; for now treat `show_error` as a magnifying glass.)

In [1]:
import sys
print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 10), "Upgrade to Python 3.10+ (match-case appears later in Basics)"

def show_error(code: str):
    """Run one line and print the result - or the error, without stopping the notebook."""
    try:
        print(f"{code}  ->  {eval(code, globals())!r}")
    except Exception as e:
        print(f"{code}  ->  💥 {type(e).__name__}: {e}")

Python 3.12.3


---
## 1. Names point to objects

Watch what happens to two names when one of them moves:

In [2]:
a = 10
b = a            # a second name tag on the SAME object
a = 20           # the 'a' tag moves to a new object; 'b' is untouched
print("a =", a, "| b =", b)
print("same object?", id(a) == id(b))

a = 20 | b = 10
same object? False


Numbers can't be changed in place, so you never *felt* the tags. **Lists can** — and that's where the box picture betrays you:

In [3]:
morning_shift = ["Asha", "Ravi"]
evening_shift = morning_shift            # NOT a copy - a second tag on the same list
evening_shift.append("Kabir")
print("morning :", morning_shift)        # Kabir shows up here too!
print("same object?", morning_shift is evening_shift)

evening_shift = morning_shift.copy()     # a real, separate copy
evening_shift.append("Meera")
print("morning :", morning_shift, "| evening :", evening_shift)

morning : ['Asha', 'Ravi', 'Kabir']
same object? True
morning : ['Asha', 'Ravi', 'Kabir'] | evening : ['Asha', 'Ravi', 'Kabir', 'Meera']


This splits every Python type into two families — and it explains a huge amount of behaviour later (function arguments, dictionary keys, default values):

| Family | Types | What "changing it" means |
|---|---|---|
| **Immutable** | `int`, `float`, `str`, `bool`, `None`, `tuple` | Impossible in place — Python builds a *new* object and moves your tag |
| **Mutable** | `list`, `dict`, `set` | Changed *in place* — **every** tag on it sees the change |

---
## 2. The five types you'll touch every day

| Type | Example | Iron Gym use | Careful with |
|---|---|---|---|
| `int` | `25`, `6` | age, months, visit count | no size limit in Python — it just grows |
| `float` | `1499.50` | prices, BMI | binary rounding errors (see the Operators notebook) |
| `str` | `"Rahul"` | names, plans, phone numbers | `"25"` is **text**, not a number |
| `bool` | `True` / `False` | `is_active`, `has_paid` | it's secretly an `int` (below) |
| `None` | `None` | "no trainer assigned yet" | not the same as `0`, `""` or `False` |

In [4]:
member_name = "Rahul Kumar"
age         = 25
monthly_fee = 1499.50
is_active   = True
trainer     = None          # None = "no value (yet)" - NOT 0 and NOT ""

for value in (member_name, age, monthly_fee, is_active, trainer):
    print(f"{value!r:<15} -> {type(value).__name__}")

'Rahul Kumar'   -> str
25              -> int
1499.5          -> float
True            -> bool
None            -> NoneType


In [5]:
print(isinstance(age, int))        # the way to ask "is it an int?" (preferred over type(age) == int)
print(isinstance(True, int))       # True! bool is a subclass of int...
print(True + True, False * 10)     # ...so True behaves as 1 and False as 0

True
True
2 0


---
## 3. Conversion — where bugs are born

Data almost never arrives in the type you want. A form box, `input()`, a CSV cell, an API field — **everything arrives as text**. The professional habit: **convert once, at the edge**, then never think about strings again.

In [6]:
# In a real script:   raw_age = input("Age? ")   -> ALWAYS a str, even if the user types 25
raw_age = "25"                     # simulating what input() hands you

show_error("raw_age + 1")          # the front-desk crash from the top of this notebook
print("next year:", int(raw_age) + 1)

raw_age + 1  ->  💥 TypeError: can only concatenate str (not "int") to str
next year: 26


In [7]:
show_error('int("25")')
show_error('int("25.5")')          # int() refuses text that has a decimal point
show_error('int(float("25.5"))')   # two steps: text -> float -> int
show_error('int(25.9)')            # truncates toward zero. It does NOT round!
show_error('round(25.9)')
show_error('float("1,499.50")')    # commas break it - clean the text first
show_error('int("  25  ")')        # surrounding spaces are fine
show_error('int("")')              # empty text is not zero
show_error('str(25) + " years"')

int("25")  ->  25
int("25.5")  ->  💥 ValueError: invalid literal for int() with base 10: '25.5'
int(float("25.5"))  ->  25
int(25.9)  ->  25
round(25.9)  ->  26
float("1,499.50")  ->  💥 ValueError: could not convert string to float: '1,499.50'
int("  25  ")  ->  25
int("")  ->  💥 ValueError: invalid literal for int() with base 10: ''
str(25) + " years"  ->  '25 years'


---
## 4. Truthy and falsy — every object can answer "yes or no?"

`bool(x)` is what `if x:` uses behind the scenes. The rule is short: **zero, empty, and `None` are falsy. Everything else is truthy.**

In [8]:
for value in [0, 0.0, "", [], None, False, "0", " ", [0], "False"]:
    print(f"{value!r:>8}  ->  {bool(value)}")

       0  ->  False
     0.0  ->  False
      ''  ->  False
      []  ->  False
    None  ->  False
   False  ->  False
     '0'  ->  True
     ' '  ->  True
     [0]  ->  True
 'False'  ->  True


Notice the last four: the *text* `"0"` and `"False"` are truthy, and `[0]` is a non-empty list. That's why `bool(input())` is useless for yes/no questions — any typed text, even `"no"`, is truthy.

---
## 5. Formatting output: f-strings

You'll write `f"..."` thousands of times. The part after the colon controls how a value is displayed:

| Spec | Meaning | Example → result |
|---|---|---|
| `:.2f` | 2 decimals | `f"{1499.5:.2f}"` → `1499.50` |
| `:,` | thousands separator | `f"{1499500:,}"` → `1,499,500` |
| `:>10` / `:<10` / `:^10` | right / left / centre in 10 chars | `f"{'Pro':>10}"` → `       Pro` |
| `:05d` | zero-pad to 5 digits | `f"{42:05d}"` → `00042` |
| `:.1%` | percent | `f"{0.125:.1%}"` → `12.5%` |
| `{x=}` | print name **and** value (debugging gold) | `f"{age=}"` → `age=25` |

In [9]:
name, fee, months, discount = "Rahul", 14995.5, 6, 0.125
print(f"{name:<10}|{fee:>12,.2f}|")     # left-aligned name, right-aligned money with separators
print(f"{months:03d} months")           # zero-padded
print(f"discount: {discount:.1%}")      # percent
print(f"{name=}  {months=}")            # debugging shortcut
print(f"{'Plan':^20}")                  # centred
print(f"{'':-^20}")                     # a divider line made of '-'

Rahul     |   14,995.50|
006 months
discount: 12.5%
name='Rahul'  months=6
        Plan        
--------------------


---
## 6. Naming rules, constants, and two handy tricks

| ✅ Valid | ❌ Invalid | Why |
|---|---|---|
| `monthly_fee`, `_temp`, `fee2` | `2fast` | can't start with a digit |
| `member_name` | `member-name` | `-` is subtraction |
| `plan_type` | `class`, `if`, `for` | reserved keywords |

Convention (PEP 8): `snake_case` for variables and functions, `UPPER_CASE` for constants. Python won't *stop* you changing a constant — the capitals are a promise to your future self.

In [10]:
GST_PERCENT = 18                          # constant by convention
joining_fee = registration_fee = 500      # chained assignment (fine for numbers; risky for lists!)

a, b = 1, 2
a, b = b, a                               # swap with no temp variable
print(a, b)

first, *rest = ["Asha", "Ravi", "Kabir"]  # unpacking: first one + "everything else"
print(first, rest)

import keyword
print(len(keyword.kwlist), "reserved words, e.g.", keyword.kwlist[:6])

2 1
Asha ['Ravi', 'Kabir']
35 reserved words, e.g. ['False', 'None', 'True', 'and', 'as', 'assert']


---
## Real-world capstone: the Iron Gym membership receipt

Raw text goes in (exactly as a form or `input()` would deliver it), conversion happens **once at the edge**, and a clean receipt comes out. Every concept above shows up: tags, types, conversion, f-string formatting.

In [11]:
# --- raw inputs: everything is a STRING, messy on purpose ---
raw_name, raw_plan, raw_months, raw_price, raw_discount = "  rAHUL kumar ", "pro", "6", "1499.50", "10"

# --- convert at the edge ---
name     = raw_name.strip().title()
plan     = raw_plan.upper()
months   = int(raw_months)
price    = float(raw_price)
discount = float(raw_discount) / 100
GST_RATE = 0.18

# --- calculate ---
subtotal     = price * months
discount_amt = subtotal * discount
taxable      = subtotal - discount_amt
gst          = taxable * GST_RATE
total        = taxable + gst
is_long_term = months >= 6

# --- display ---
print(f"{'IRON GYM':^34}")
print(f"{'':=^34}")
print(f"{'Member':<14}{name:>20}")
print(f"{'Plan':<14}{plan:>20}")
print(f"{'Duration':<14}{months:>17} mo")
print(f"{'':-^34}")
print(f"{'Subtotal':<14}{subtotal:>20,.2f}")
print(f"{'Discount':<14}{-discount_amt:>20,.2f}")
print(f"{'GST (18%)':<14}{gst:>20,.2f}")
print(f"{'TOTAL':<14}{total:>20,.2f}")
print(f"{'':=^34}")
print("Long-term member:", is_long_term)

             IRON GYM             
Member                 Rahul Kumar
Plan                           PRO
Duration                      6 mo
----------------------------------
Subtotal                  8,997.00
Discount                   -899.70
GST (18%)                 1,457.51
TOTAL                     9,554.81
Long-term member: True


> Floats are fine for *displaying* a receipt like this. The **Operators** notebook shows why real billing code stores money as whole paise instead.

### Try it yourself
1. Change `raw_months` to `"six"` and run it. Read the error from the bottom up — which line, which type, which value? Then fix it by *validating before converting* (you'll have the tools after the Strings notebook).
2. Predict the output of `bool("False")`, `int(True) + int("2")` and `float("3") == 3` **before** running them.
3. Make `evening_shift` and `morning_shift` independent using three *different* techniques (hint: `.copy()`, `list(...)`, a full slice `[:]`).
4. Put a `₹` in front of every money line and keep the columns aligned (hint: width counts the symbol).
5. Add a `joining_fee` that applies **once**, not per month, and shows on its own line.

---
## Quick reference

| I want to… | Use |
|---|---|
| know the type | `type(x)` / `isinstance(x, int)` |
| convert text → number | `int("25")`, `float("25.5")` — convert **once**, at the edge |
| make a real copy of a list | `x.copy()` or `list(x)` — never `y = x` |
| show money nicely | `f"{amount:,.2f}"` |
| debug a value fast | `f"{x=}"` |
| test for "no value" | `x is None` (never `x == None`) |

**Rule of thumb:** *data arrives as text — convert it once at the edge, and after that every variable should have exactly one type.*

---
## 🎯 Interview corner

**1. Is Python statically or dynamically typed?** Dynamically (types are attached to objects, not names) *and* strongly (no silent coercion like `"5" + 1`).
**2. Mutable vs immutable — why does it matter?** Mutable objects change in place, so every name pointing at them sees the change (aliasing bugs); only immutable (hashable) objects can be dictionary keys.
**3. What does `input()` return?** Always a `str`. Convert with `int()`/`float()`.
**4. Why is `bool("False")` `True`?** Any non-empty string is truthy — Python doesn't parse the text.
**5. `b = a` for a list — copy or alias?** Alias: two names, one object. Use `.copy()`.
**6. `int(25.9)` gives?** `25` — it truncates toward zero; use `round()` to round.

**Next:** `02_operators/` — arithmetic, comparison, logic, and why ₹19.99 × 100 ≠ 1999.